# 01 — Data identity and point-in-time behavior

## Learning objectives

- Load the accepted pinned GPW publication through production interfaces.
- Trace vendor/official alias evidence to stable identity, membership, bars, and later research input.
- Keep the official 60-member denominator distinct from priced and feature-eligible subsets.
- Distinguish session, event, availability, validity, decision, and execution times.
- Use bounded DuckDB, Polars, and PyArrow access and understand the measured physical-layout limits.

**Evidence examined.** HEAD `00e35d98a49492a7913a1e862117c5ae19757d06`; GPW `phaseb-f88fc2d38e9811ed1573`; U.S. metadata/profile for `phaseb-5d7086751156ac48cef3`; Phase A `phasea-2a2b3898aba37814`.

## Configuration

        Path overrides are environment variables; no canonical path is written.

In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd

REPO_ROOT = Path(os.environ.get("ATS_REPO_ROOT", r"D:\Stock\ATS"))
DATA_ROOT = Path(os.environ.get("ATS_DATA_ROOT", r"D:\Stock\data\ATS"))
PROJECT_ROOT = REPO_ROOT / "source" / "python"
RESEARCH_ROOT = REPO_ROOT / "RESEARCH"
GPW_MANIFEST = Path(os.environ.get(
    "ATS_GPW_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-f88fc2d38e9811ed1573" / "manifest.json",
))
US_MANIFEST = Path(os.environ.get(
    "ATS_US_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-5d7086751156ac48cef3" / "manifest.json",
))
PHASE_A_RUN = Path(os.environ.get(
    "ATS_PHASE_A_RUN", DATA_ROOT / "phase_a" / "runs" / "phasea-2a2b3898aba37814"
))
PHASE_A_EXTENDED_RUN = Path(os.environ.get(
    "ATS_PHASE_A_EXTENDED_RUN",
    DATA_ROOT / "decision_oriented_phase_a" / "runs" / "extension-20260820T163347Z",
))
PHASE_C_RUN = Path(os.environ.get(
    "ATS_PHASE_C_RUN", DATA_ROOT / "phase_c" / "runs" / "phasec-fa439d650410376aae9e"
))
PHASE_C_REPRODUCTION = Path(os.environ.get(
    "ATS_PHASE_C_REPRODUCTION",
    DATA_ROOT / "phase_c" / "reproductions" / "00e35d9" / "phasec-fa439d650410376aae9e",
))

src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

native_library_dir = str(Path(sys.prefix) / "Library" / "bin")
path_entries = [entry.rstrip("\\").lower() for entry in os.environ.get("PATH", "").split(os.pathsep)]
if native_library_dir.rstrip("\\").lower() not in path_entries:
    raise RuntimeError(f"Conda native-library directory is absent from kernel PATH: {native_library_dir}")

required = [REPO_ROOT, DATA_ROOT, GPW_MANIFEST, PHASE_A_RUN, PHASE_C_RUN]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required retained evidence is missing: {missing}")

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 140)
print({
    "repo": str(REPO_ROOT),
    "data": str(DATA_ROOT),
    "python": sys.version.split()[0],
    "native_library_path": True,
    "jupyter_runtime": os.environ.get("JUPYTER_RUNTIME_DIR"),
})

{'repo': 'D:\\Stock\\ATS', 'data': 'D:\\Stock\\data\\ATS', 'python': '3.12.13', 'native_library_path': True, 'jupyter_runtime': 'D:\\Stock\\ATS\\RESEARCH\\.tmp\\ats-env\\jupyter'}


## Validate and identify the publication

`ats_data.validate_manifest` independently checks the exact schema, declared files, hashes, row counts, configuration/version identity, implementation provenance, and manifest boundaries. It does not follow a `latest` pointer.

In [2]:
from ats_data import validate_manifest
from ats_data.discovery import manifest_files, scan_table

manifest = validate_manifest(GPW_MANIFEST)
table_summary = pd.DataFrame([
    {
        "table": table.table_name,
        "rows": table.rows,
        "files": len(table.files),
        "row_groups": sum(item.row_groups for item in table.files),
        "bytes": sum(item.bytes for item in table.files),
        "schema_version": table.schema_version,
        "semantic_key": ", ".join(table.semantic_key),
    }
    for table in manifest.tables
])
print({
    "dataset_version_id": manifest.dataset_version_id,
    "manifest_hash": manifest.manifest_hash,
    "implementation_commit": manifest.implementation_provenance["commit"],
    "implementation_clean": manifest.implementation_provenance["clean"],
})
display(table_summary)

{'dataset_version_id': 'phaseb-f88fc2d38e9811ed1573', 'manifest_hash': 'dd0d37e913342002aa208cd85e4a1fcbbb597b608c3abd08d7bead94ef9ec4aa', 'implementation_commit': '94a0e6e0792937a4b6ec8dc69c66fca85908a877', 'implementation_clean': True}


,table,rows,files,row_groups,bytes,schema_version,semantic_key
0,bars,147687,1,2,4069116,ats.canonical.v2,"security_id, event_ts, frequency, source, adju..."
1,security_aliases,2039,1,1,18321,ats.canonical.v2,"security_id, identifier_type, identifier_value..."
2,security_master,93,1,1,6918,ats.canonical.v2,security_id
3,universe_membership,1760,1,1,17770,ats.canonical.v2,"universe_id, universe_component, raw_identifie..."


**Interpretation.** The logical version ID is content/configuration derived; the physical evidence is the manifest's explicit Parquet file list with byte and logical hashes. A file path alone is not a dataset identity, and a mutable discovery pointer is not accepted as an execution input.

## Schemas and manifest-listed physical files

PyArrow inspects the exact stored schema. Only a short projection is displayed; contracts in `ats_contracts.schemas` define the full expected order, types, nullability, and semantic keys.

In [3]:
import pyarrow.parquet as pq

physical = []
for table_name in ["security_master", "security_aliases", "universe_membership", "bars"]:
    for path in manifest_files(GPW_MANIFEST, table_name):
        parquet = pq.ParquetFile(path)
        physical.append({
            "table": table_name,
            "relative_path": str(path.relative_to(GPW_MANIFEST.parent)),
            "rows": parquet.metadata.num_rows,
            "row_groups": parquet.metadata.num_row_groups,
            "columns": parquet.schema_arrow.names[:6],
        })
display(pd.DataFrame(physical))
print("bars schema (first 10 fields):")
print(pq.read_schema(manifest_files(GPW_MANIFEST, "bars")[0]).names[:10])

,table,relative_path,rows,row_groups,columns
0,security_master,data\security_master\market=GPW\part-000.parquet,93,1,"[security_id, issuer_id, market, venue_mic, in..."
1,security_aliases,data\security_aliases\market=GPW\part-000.parquet,2039,1,"[security_id, identifier_type, identifier_valu..."
2,universe_membership,data\universe_membership\part-000.parquet,1760,1,"[universe_id, universe_component, security_id,..."
3,bars,data\bars\market=GPW\frequency=daily\part-000....,147687,2,"[security_id, market, venue_mic, frequency, ev..."


bars schema (first 10 fields):
['security_id', 'market', 'venue_mic', 'frequency', 'event_ts', 'session_date', 'available_ts', 'open', 'high', 'low']


**Interpretation.** Canonical bars are compact and security/event sorted: one GPW bars file and two row groups, with no ticker/security/year/month partitions. Projection and predicates can be pushed into readers; physical clustering may prune row groups but is not a semantic contract.

## In-memory DuckDB catalog from the pinned file list

This uses production `manifest_files` to resolve only declared files, then creates temporary in-memory views. It demonstrates one-session, one-security, and bounded-range access without writing a catalog.

In [4]:
from datetime import date
import duckdb

con = duckdb.connect()
for table_name in ["bars", "security_master", "security_aliases", "universe_membership"]:
    paths = [str(path) for path in manifest_files(GPW_MANIFEST, table_name)]
    con.from_parquet(paths, union_by_name=False).create_view(table_name)

session = date(2022, 11, 4)
representative = con.execute("""
    SELECT b.security_id, count(*) AS history_rows, min(b.session_date) AS first_session,
           max(b.session_date) AS last_session
    FROM bars b
    JOIN (
      SELECT DISTINCT security_id FROM universe_membership
      WHERE valid_from <= ? AND (valid_to IS NULL OR valid_to >= ?)
        AND security_id IS NOT NULL
    ) m USING (security_id)
    GROUP BY b.security_id
    ORDER BY history_rows DESC, b.security_id
    LIMIT 1
""", [session, session]).df().iloc[0]
security_id = representative["security_id"]
display(pd.DataFrame([representative]))

history = con.execute("""
    SELECT session_date, open, close, volume, event_ts, available_ts
    FROM bars WHERE security_id = ? AND session_date BETWEEN DATE '2022-10-28' AND DATE '2022-11-10'
    ORDER BY session_date
""", [security_id]).df()
display(history)

,security_id,history_rows,first_session,last_session
0,002d9386-d85e-55c7-8ee3-2b6c2d073fb7,1750,2019-01-02,2025-12-30


,session_date,open,close,volume,event_ts,available_ts
0,2022-10-28,117.291,121.708,614799.730163,2022-10-28 17:00:00+02:00,2022-10-28 17:05:00+02:00
1,2022-10-31,121.708,124.750,451672.277503,2022-10-31 17:00:00+01:00,2022-10-31 17:05:00+01:00
2,2022-11-02,125.143,133.702,935740.400491,2022-11-02 17:00:00+01:00,2022-11-02 17:05:00+01:00
3,2022-11-03,130.542,131.758,515351.402404,2022-11-03 17:00:00+01:00,2022-11-03 17:05:00+01:00
4,2022-11-04,132.504,136.608,368413.212403,2022-11-04 17:00:00+01:00,2022-11-04 17:05:00+01:00
5,2022-11-07,137.393,133.761,442662.733785,2022-11-07 17:00:00+01:00,2022-11-07 17:05:00+01:00
6,2022-11-08,133.487,133.565,439652.081100,2022-11-08 17:00:00+01:00,2022-11-08 17:05:00+01:00
7,2022-11-09,133.192,133.800,197789.183617,2022-11-09 17:00:00+01:00,2022-11-09 17:05:00+01:00
8,2022-11-10,133.387,138.197,401372.472755,2022-11-10 17:00:00+01:00,2022-11-10 17:05:00+01:00


**Interpretation.** The query is deliberately bounded. The selected identity has sufficient retained history, and the result shows daily session dates alongside modeled close event/availability timestamps. No giant table was materialized.

## Identity trace: alias evidence → stable security → membership → bar

Aliases carry validity and provenance. The stable `security_id` owns continuity; tickers and vendor identifiers do not.

In [5]:
master = con.execute("SELECT * FROM security_master WHERE security_id = ?", [security_id]).df()
aliases = con.execute("""
    SELECT identifier_type, identifier_value, raw_identifier, vendor, valid_from, valid_to,
           resolution_status, provenance
    FROM security_aliases WHERE security_id = ?
    ORDER BY identifier_type, valid_from NULLS FIRST
""", [security_id]).df()
membership = con.execute("""
    SELECT universe_id, universe_component, raw_identifier, valid_from, valid_to,
           resolution_status, member_state, official_denominator
    FROM universe_membership WHERE security_id = ?
    ORDER BY valid_from
""", [security_id]).df()
bar = con.execute("""
    SELECT security_id, session_date, open, close, source, source_record_id,
           adjustment_state, resolution_state
    FROM bars WHERE security_id = ? AND session_date = ?
""", [security_id, session]).df()
display(master[["security_id", "market", "instrument_type", "identity_status", "status", "source"]])
display(aliases.head(8))
display(membership.head(8))
display(bar)

,security_id,market,instrument_type,identity_status,status,source
0,002d9386-d85e-55c7-8ee3-2b6c2d073fb7,GPW,common_equity,authoritative,known_official_member,trusted_phase_a_identity


,identifier_type,identifier_value,raw_identifier,vendor,valid_from,valid_to,resolution_status,provenance
0,isin,PLOPTTC00011,PLOPTTC00011,NaN,2020-11-27,2026-03-22,resolved,official snapshot ISIN / configured listing venue
1,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2020-11-27,2020-12-20,resolved,reference/gpw_indices/snapshots/WIG20/2020-11-...
2,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2020-12-21,2021-03-21,resolved,reference/gpw_indices/snapshots/WIG20/2020-12-...
3,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2021-03-22,2021-06-20,resolved,reference/gpw_indices/snapshots/WIG20/2021-03-...
4,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2021-06-21,2021-09-19,resolved,reference/gpw_indices/snapshots/WIG20/2021-06-...
5,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2021-09-20,2021-12-19,resolved,reference/gpw_indices/snapshots/WIG20/2021-09-...
6,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2021-12-20,2022-03-20,resolved,reference/gpw_indices/snapshots/WIG20/2021-12-...
7,official_short_name,CDPROJEKT,CDPROJEKT,NaN,2022-03-21,2022-06-19,resolved,reference/gpw_indices/snapshots/WIG20/2022-03-...


,universe_id,universe_component,raw_identifier,valid_from,valid_to,resolution_status,member_state,official_denominator
0,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2020-11-27,2020-12-20,official_isin_resolved,official_resolved,60
1,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2020-12-21,2021-03-21,official_isin_resolved,official_resolved,60
2,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2021-03-22,2021-06-20,official_isin_resolved,official_resolved,60
3,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2021-06-21,2021-09-19,official_isin_resolved,official_resolved,60
4,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2021-09-20,2021-12-19,official_isin_resolved,official_resolved,60
5,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2021-12-20,2022-03-20,official_isin_resolved,official_resolved,60
6,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2022-03-21,2022-06-19,official_isin_resolved,official_resolved,60
7,GPW_TOP60_WIG20_MWIG40,WIG20,PLOPTTC00011,2022-06-20,2022-08-03,official_isin_resolved,official_resolved,60


,security_id,session_date,open,close,source,source_record_id,adjustment_state,resolution_state
0,002d9386-d85e-55c7-8ee3-2b6c2d073fb7,2022-11-04,132.504,136.608,stooq_local_bulk,daily/pl/wse stocks/cdr.txt#CDR#20221104,vendor_adjusted_semantics_unverified,resolved


**Interpretation.** Raw/vendor/official identifiers remain evidence attached to a stable key. Validity intervals answer *when an assertion applies*. They do not manufacture announcement/availability timestamps when the source did not retain them.

## Official denominator versus usable populations

The representative 2022-11-04 session is intentionally useful: the official universe and resolved identities are 60, while only 57 have usable prices and 56 have complete momentum history. Missing-price members remain official rows with explicit benign-exit states.

In [6]:
active = con.execute("""
    SELECT raw_identifier, security_id, resolution_status, member_state, official_denominator
    FROM universe_membership
    WHERE valid_from <= ? AND (valid_to IS NULL OR valid_to >= ?)
    ORDER BY raw_identifier
""", [session, session]).df()
priced = con.execute("""
    SELECT DISTINCT m.security_id
    FROM universe_membership m JOIN bars b
      ON b.security_id = m.security_id AND b.session_date = ?
    WHERE m.valid_from <= ? AND (m.valid_to IS NULL OR m.valid_to >= ?)
""", [session, session, session]).df()

panel_path = PHASE_A_RUN / "artifacts" / "research_panel.parquet"
panel_columns = [
    "session_date", "isin", "raw_identifier", "security_id", "price_eligibility_state",
    "price_exclusion_reason", "is_price_usable_member", "price_usable_member_count",
    "is_feature_eligible__momentum_12_1__v1",
    "feature_exclusion_reason__momentum_12_1__v1",
    "feature_usable_member_count__momentum_12_1__v1", "official_member_count",
]
cross = pd.read_parquet(panel_path, columns=panel_columns)
cross = cross[pd.to_datetime(cross["session_date"]).dt.date == session].copy()

population = pd.DataFrame([{
    "official_expected": len(active),
    "identity_resolved": int(active["security_id"].notna().sum()),
    "canonical_priced": len(priced),
    "phase_a_price_usable": int(cross["is_price_usable_member"].sum()),
    "momentum_feature_eligible": int(cross["is_feature_eligible__momentum_12_1__v1"].sum()),
    "official_denominator": int(cross["official_member_count"].iloc[0]),
}])
display(population)
display(cross.loc[~cross["is_price_usable_member"], [
    "isin", "raw_identifier", "price_eligibility_state", "price_exclusion_reason"
]])
assert population.loc[0, "official_expected"] == 60
assert population.loc[0, "phase_a_price_usable"] == 57

,official_expected,identity_resolved,canonical_priced,phase_a_price_usable,momentum_feature_eligible,official_denominator
0,60,60,57,57,56,60


,isin,raw_identifier,price_eligibility_state,price_exclusion_reason
29102,PLPGNIG00014,PLPGNIG00014,unresolved_vendor_alias,unresolved_vendor_alias
29127,PLSTSHL00012,PLSTSHL00012,unresolved_vendor_alias,unresolved_vendor_alias
29159,PLCIECH00018,PLCIECH00018,unresolved_vendor_alias,unresolved_vendor_alias


**Interpretation.** `57/60` is a state, not a new universe. Feature eligibility can be smaller again because an exact lookback is required. Research ranks use the named feature's eligible count, while coverage reporting retains the official 60 denominator.

## Polars and PyArrow bounded access

`scan_table` refuses discovery pointers and returns a lazy scan with projection/predicate pushdown. PyArrow provides schema/dataset inspection without loading all rows.

In [7]:
import polars as pl
import pyarrow.dataset as ds

polars_cross = (
    scan_table(GPW_MANIFEST, "bars")
    .filter(pl.col("session_date") == session)
    .select("security_id", "session_date", "close", "available_ts")
    .sort("security_id")
    .limit(5)
    .collect()
)
display(polars_cross)

dataset = ds.dataset([str(p) for p in manifest_files(GPW_MANIFEST, "bars")], format="parquet")
print({"dataset_schema_fields": len(dataset.schema), "projected_fields": dataset.schema.names[:8]})

security_id,session_date,close,available_ts
str,date,f64,"datetime[μs, UTC]"
"""002d9386-d85e-55c7-8ee3-2b6c2d…",2022-11-04,136.608,2022-11-04 16:05:00 UTC
"""029e761a-f8f2-59df-a3dd-b5e104…",2022-11-04,25.8,2022-11-04 16:05:00 UTC
"""041f6696-97c3-5f6f-9b82-8cab30…",2022-11-04,4.96454,2022-11-04 16:05:00 UTC
"""0ab1d3f3-e236-5083-97c6-43b2d1…",2022-11-04,4.432,2022-11-04 16:05:00 UTC
"""0e09a7cb-8b88-5e91-9b32-de6128…",2022-11-04,18.57,2022-11-04 16:05:00 UTC


{'dataset_schema_fields': 24, 'projected_fields': ['security_id', 'market', 'venue_mic', 'frequency', 'event_ts', 'session_date', 'available_ts', 'open']}


**Interpretation.** DuckDB, Polars, and PyArrow are readers over one canonical Parquet publication, not competing canonical copies.

## Point-in-time vocabulary and a concrete lookahead

| Concept | Meaning here | Owner |
|---|---|---|
| session date | Local exchange trading session | canonical bars/membership |
| event time | Modeled completion of the daily bar | canonical bar |
| availability time | Earliest modeled time the completed bar may be used | canonical bar, propagated into research |
| validity interval | Dates on which identity/membership assertion applies | canonical identity/membership |
| decision visibility | Inputs with `available_ts <= decision_ts` | Phase A/research |
| next eligible session/open | First later session allowed by the intent and calendar | Phase C |
| Phase C execution time | Modeled exchange open; only open is visible | Phase C market timing policy |

For the retained pre-open Phase A decision, features use the *prior* session's close. `close[t]` occurs after the 08:45 decision and is lookahead if used to form that signal. The accepted labels deliberately start at `close[t]`: they are diagnostic future outcomes, not same-close executable returns. Phase C requires `information_available_ts <= decision_ts < eligible_open` and fills only at a later modeled open.

In [8]:
sample = cross.loc[cross["is_price_usable_member"]].iloc[0]
temporal = pd.read_parquet(
    panel_path,
    columns=["session_date", "feature_session_date", "feature_event_ts", "feature_available_ts", "decision_ts", "label_start_close"],
)
temporal = temporal[pd.to_datetime(temporal["session_date"]).dt.date == session].iloc[0]
display(pd.DataFrame([temporal]))
assert temporal["feature_available_ts"] <= temporal["decision_ts"]
print("Lookahead example: label_start_close belongs to decision session close and is not visible at the pre-open decision.")

,session_date,feature_session_date,feature_event_ts,feature_available_ts,decision_ts,label_start_close
29100,2022-11-04,2022-11-03,2022-11-03 17:00:00+01:00,2022-11-03 17:05:00+01:00,2022-11-04 08:45:00+01:00,136.608


Lookahead example: label_start_close belongs to decision session close and is not visible at the pre-open decision.


## Why Phase B exists, and its measured physical limits

Phase B hardened exact schemas, stable identity, unresolved-state preservation, temporal fields, immutable versioning, transactional publication, and exact Phase A→B equivalence. GPW reconciliation preserves semantic-key/numeric hashes, identity/membership hashes, and denominator/usable counts at zero numeric tolerance.

Retained profiles show GPW one-security access considered 1 of 2 row groups; U.S. one-security access considered 1 of 252. Both readers expose projection/predicate pushdown. The compact one-file layout avoids file explosion, but time-window pruning is limited because data is security-first sorted and not time partitioned. These are measured properties, not a request to optimize this checkpoint.

In [9]:
gpw_profile = json.loads((DATA_ROOT / "phase_b" / "reports" / "gpw_profile.json").read_text(encoding="utf-8"))
us_profile = json.loads((DATA_ROOT / "phase_b" / "reports" / "us_profile.json").read_text(encoding="utf-8"))
profiles = pd.DataFrame([
    {
        "market": market,
        "version": profile["dataset_version_id"],
        "bars": profile["tables"]["bars"]["rows"],
        "bar_files": profile["tables"]["bars"]["files"],
        "total_row_groups": profile["row_group_pruning_evidence"]["total_row_groups"],
        "candidate_groups_one_security": profile["row_group_pruning_evidence"]["one_security_candidate_row_groups"],
        "predicate_pushdown_visible": profile["polars"]["predicate_pushdown_visible"],
    }
    for market, profile in [("GPW", gpw_profile), ("US", us_profile)]
])
display(profiles)

,market,version,bars,bar_files,total_row_groups,candidate_groups_one_security,predicate_pushdown_visible
0,GPW,phaseb-f88fc2d38e9811ed1573,147687,1,2,1,True
1,US,phaseb-5d7086751156ac48cef3,30937812,1,252,1,True


**Interpretation.** The U.S. example remains metadata-only. All 15,355 U.S. listing identities are source-scoped provisional identities with null issuer IDs; 137 ingestion issues remain visible. Authoritative issuer mapping, corporate actions, and validity metadata are not established.

## Safe to rely on now

- The exact pinned GPW publication, manifest contracts/hashes, bounded readers, stable GPW identities, and preserved official denominator.

## Usable with documented caveats

- Modeled daily event/availability times and incomplete GPW membership announcement history.
- U.S. canonical facts for source-scoped analysis, while identity/issuer/action resolution remains provisional.
- Current security-first Parquet layout: measured and functional, with known time-pruning limits.

## Not implemented or not safe to rely on

- Mutable discovery pointers in experiments; invented ISIN/ticker continuity; authoritative U.S. issuer or corporate-action history; exact live auction availability.